# Module 6: Taxonomy Deep-Dive (AGORA-Specific Value)

**Question:** What does the AGORA taxonomy reveal about legislative strategy?

This module exploits unique AGORA annotation columns not available in standard Congress.gov datasets.

Analyses:
1. Strategy profiles by community
2. Harm → strategy legislative playbook
3. Application coverage gaps
4. Incentive structures by party
5. Risk factor attention by sponsor

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath(".")))

import pandas as pd
import numpy as np
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from collections import defaultdict

import analysis_utils as au

docs_df = au.load_comprehensive_df()
cosp_df = au.load_cosponsors_df()
communities = au.load_communities()

print(f"Documents: {len(docs_df)}")
print(f"Taxonomy groups available: {list(au.TAXONOMY_GROUPS.keys())}")
for g in au.TAXONOMY_GROUPS:
    cols = au.get_taxonomy_columns(docs_df, g)
    print(f"  {g}: {len(cols)} tags")

## 6.1 Strategy Profiles by Community

Which communities favor disclosure vs. evaluation vs. input controls vs. new institutions? Each community's aggregate taxonomy vector reveals its regulatory philosophy.

In [ ]:
# Build community → document mapping via cosponsor membership
cosp_policy = cosp_df.merge(docs_df[["AGORA ID"]], on="AGORA ID", how="inner")

# For each community, gather all documents its members cosponsor
comm_docs = {}
for i, comm in enumerate(communities):
    members = set(comm.get("members", []))
    aids = set(cosp_policy[cosp_policy["Cosponsor_BioguideId"].isin(members)]["AGORA ID"])
    comm_docs[f"Community {i}"] = aids

# Aggregate strategy tags per community (use top-level strategy categories)
strat_matrix = au.taxonomy_vector(docs_df, "Strategies")
strat_matrix["AGORA ID"] = docs_df["AGORA ID"].values

comm_strat = {}
for comm_name, aids in comm_docs.items():
    mask = strat_matrix["AGORA ID"].isin(aids)
    comm_strat[comm_name] = strat_matrix.loc[mask].drop(columns="AGORA ID").sum()

comm_strat_df = pd.DataFrame(comm_strat).T
# Normalize to proportions within each community
comm_strat_norm = comm_strat_df.div(comm_strat_df.sum(axis=1), axis=0).fillna(0)

# Filter to top-level strategies (not sub-strategies) for readability
top_level = [c for c in comm_strat_norm.columns if ":" not in c and len(c) > 2]
if len(top_level) < 5:
    top_level = comm_strat_norm.sum().nlargest(15).index.tolist()

fig_strat = px.imshow(
    comm_strat_norm[top_level].values,
    x=[c[:30] for c in top_level],
    y=comm_strat_norm.index.tolist(),
    color_continuous_scale="Viridis",
    labels={"color": "Proportion"},
    title="Community Regulatory Philosophy: Strategy Profiles",
    aspect="auto",
)
fig_strat.update_layout(template="plotly_white", height=350, width=1000,
                          xaxis_tickangle=-40)
fig_strat.show()

## 6.2 Harm → Strategy Legislative Playbook

When legislators address a specific harm (e.g., discrimination, privacy violation), which regulatory strategies do they reach for? This maps the implicit policy theory of Congress.

In [ ]:
# Build Harm → Strategy co-occurrence as a Sankey diagram
harm_vec = au.taxonomy_vector(docs_df, "Harms")
strat_vec = au.taxonomy_vector(docs_df, "Strategies")

# Get top-level strategies only for readability
strat_top = [c for c in strat_vec.columns if ":" not in c and len(c) > 2]
if len(strat_top) < 5:
    strat_top = strat_vec.sum().nlargest(12).index.tolist()

# Build Sankey flows: harm → strategy with document count
sankey_data = []
for harm_col in harm_vec.columns:
    harm_mask = harm_vec[harm_col].astype(bool)
    if harm_mask.sum() < 5:
        continue
    for strat_col in strat_top:
        both = (harm_mask & strat_vec[strat_col].astype(bool)).sum()
        if both > 3:
            sankey_data.append({
                "harm": harm_col[:35],
                "strategy": strat_col[:30],
                "count": int(both),
            })

sankey_df = pd.DataFrame(sankey_data)

# Build Sankey node/link lists
all_labels = list(set(sankey_df["harm"])) + list(set(sankey_df["strategy"]))
label_idx = {l: i for i, l in enumerate(all_labels)}

fig_sankey = go.Figure(go.Sankey(
    node=dict(
        label=all_labels,
        color=["#e41a1c"] * len(set(sankey_df["harm"])) + ["#377eb8"] * len(set(sankey_df["strategy"])),
        pad=15, thickness=20,
    ),
    link=dict(
        source=[label_idx[r["harm"]] for _, r in sankey_df.iterrows()],
        target=[label_idx[r["strategy"]] for _, r in sankey_df.iterrows()],
        value=[r["count"] for _, r in sankey_df.iterrows()],
    ),
))
fig_sankey.update_layout(
    title="Legislative Playbook: Harm → Strategy Mapping<br><sub>Red = Harms addressed, Blue = Strategies deployed</sub>",
    template="plotly_white", height=550, width=900,
)
fig_sankey.show()

## 6.3 Application Coverage Gaps

Which AI application domains are well-legislated vs. under-addressed? Radar chart per community shows legislative attention distribution.

In [ ]:
# Application coverage across all documents
app_vec = au.taxonomy_vector(docs_df, "Applications")
app_vec["AGORA ID"] = docs_df["AGORA ID"].values

# Overall coverage
overall_coverage = app_vec.drop(columns="AGORA ID").sum().sort_values(ascending=False)

fig_app_bar = px.bar(
    x=overall_coverage.values,
    y=[c[:40] for c in overall_coverage.index],
    orientation="h",
    labels={"x": "Number of Documents", "y": "Application Domain"},
    title="AI Application Domain Coverage in Congressional Legislation",
    color=overall_coverage.values,
    color_continuous_scale="Oranges",
)
fig_app_bar.update_layout(template="plotly_white", height=500, width=800,
                           yaxis=dict(autorange="reversed"))
fig_app_bar.show()

# Radar chart per community
comm_app = {}
for comm_name, aids in comm_docs.items():
    mask = app_vec["AGORA ID"].isin(aids)
    comm_app[comm_name] = app_vec.loc[mask].drop(columns="AGORA ID").sum()

comm_app_df = pd.DataFrame(comm_app).T
# Normalize
comm_app_norm = comm_app_df.div(comm_app_df.sum(axis=1), axis=0).fillna(0)

# Use top 10 application domains for radar
top_apps = overall_coverage.head(10).index.tolist()
radar_df = comm_app_norm[top_apps]

fig_radar = go.Figure()
colors = px.colors.qualitative.Set2
for i, (comm_name, row) in enumerate(radar_df.iterrows()):
    fig_radar.add_trace(go.Scatterpolar(
        r=row.values.tolist() + [row.values[0]],
        theta=[c[:25] for c in top_apps] + [top_apps[0][:25]],
        name=comm_name,
        line=dict(color=colors[i % len(colors)]),
        fill="toself", opacity=0.3,
    ))

fig_radar.update_layout(
    title="Application Domain Focus by Community (Normalized)",
    template="plotly_white", height=500, width=700,
    polar=dict(radialaxis=dict(visible=True, range=[0, radar_df.max().max() * 1.1])),
)
fig_radar.show()

## 6.4 Incentive Structures by Party

Criminal liability vs. civil liability vs. fines vs. subsidies: who proposes what? Do Democrats and Republicans differ in their enforcement preferences?

In [ ]:
# Incentive tags per cosponsor, grouped by party
incent_profile = au.sponsor_taxonomy_profile(docs_df, cosp_df, "Incentives")

# Add party info
bio_party = cosp_df.drop_duplicates("Cosponsor_BioguideId").set_index("Cosponsor_BioguideId")["Cosponsor_Party"]
incent_profile["party"] = incent_profile.index.map(bio_party)

# Aggregate by party
party_incent = incent_profile.groupby("party").sum(numeric_only=True)
# Normalize to proportions
party_incent_norm = party_incent.div(party_incent.sum(axis=1), axis=0)

fig_incent = px.bar(
    party_incent_norm.T.reset_index().melt(id_vars="index", var_name="Party", value_name="Proportion"),
    x="index", y="Proportion", color="Party",
    barmode="group",
    color_discrete_map={"D": "#2166ac", "R": "#b2182b", "I": "#4daf4a"},
    labels={"index": "Incentive Type", "Proportion": "Proportion of Total"},
    title="Enforcement Preferences by Party: How Do Parties Differ on Incentives?",
)
fig_incent.update_layout(template="plotly_white", height=400, width=800,
                          xaxis_tickangle=-20)
fig_incent.show()

# Raw counts for context
print("Raw incentive counts by party:")
party_incent

## 6.5 Risk Factor Attention

How much legislative attention does each risk factor get? Who are the champions of each risk area?

In [ ]:
# Risk factor profile per sponsor
risk_profile = au.sponsor_taxonomy_profile(docs_df, cosp_df, "Risk Factors")
risk_profile["party"] = risk_profile.index.map(bio_party)

# Overall risk factor attention (document count)
risk_vec = au.taxonomy_vector(docs_df, "Risk Factors")
risk_totals = risk_vec.sum().sort_values(ascending=False)

# Treemap: risk factor attention
risk_tree_data = []
for rf, count in risk_totals.items():
    # Determine if sub-category
    parts = rf.split(": ", 1)
    parent = parts[0] if len(parts) > 1 else "Risk Factors"
    risk_tree_data.append({"risk_factor": rf, "parent": parent, "documents": int(count)})

risk_tree_df = pd.DataFrame(risk_tree_data)

fig_risk_tree = px.treemap(
    risk_tree_df, path=["parent", "risk_factor"], values="documents",
    color="documents", color_continuous_scale="Reds",
    title="Risk Factor Attention: Legislative Coverage Treemap",
)
fig_risk_tree.update_layout(template="plotly_white", height=500, width=800)
fig_risk_tree.show()

# Top 3 champion sponsors per risk factor
print("Top 3 Sponsors per Risk Factor (by document count):")
for rf in risk_profile.columns:
    if rf == "party":
        continue
    top3 = risk_profile[rf].nlargest(3)
    if top3.sum() == 0:
        continue
    bio_name = cosp_df.drop_duplicates("Cosponsor_BioguideId").set_index("Cosponsor_BioguideId")["Cosponsor_FullName"]
    names = [f"{bio_name.get(b, b)[:30]} ({risk_profile.loc[b, 'party']})" for b in top3.index]
    print(f"  {rf}: {', '.join(names)}")

## 6.6 Exports

In [ ]:
au.save_df(comm_strat_norm, "community_strategy_profiles")
au.save_df(party_incent, "incentives_by_party")
au.save_df(risk_tree_df, "risk_factor_coverage")

fig_strat.write_html(str(au.OUTPUTS_DIR / "fig_community_strategies.html"))
fig_sankey.write_html(str(au.OUTPUTS_DIR / "fig_harm_strategy_sankey.html"))
fig_app_bar.write_html(str(au.OUTPUTS_DIR / "fig_application_coverage.html"))
fig_radar.write_html(str(au.OUTPUTS_DIR / "fig_application_radar.html"))
fig_incent.write_html(str(au.OUTPUTS_DIR / "fig_incentives_by_party.html"))
fig_risk_tree.write_html(str(au.OUTPUTS_DIR / "fig_risk_treemap.html"))

print("Module 6 exports saved to notebooks/outputs/")